采样参数对模型的影响

In [46]:
from dotenv import load_dotenv
import zai
from zai import ZhipuAiClient
import pandas as pd
import asyncio

In [47]:
load_dotenv()
client = ZhipuAiClient()

pd.set_option('display.max_rows', None)      # 显示所有行
pd.set_option('display.max_columns', None)   # 显示所有列
pd.set_option('display.max_colwidth', None)  # 单元格内容不截断
pd.set_option('display.width', None)         # 自动适应终端宽度

In [48]:
def fetch(params: dict) -> str:
    try:
        response = client.chat.completions.create(**params) # 异步 asyncCompletions
        content = response.choices[0].message.content
        # print(f"结果:{json.dumps(response.model_dump(), indent=2, ensure_ascii=False)}")
        return content
    except zai.core.APIStatusError as err:
        e = f"API 状态错误: {err}"
        print(e)
        return e
    except zai.core.APITimeoutError as err:
        e = f"请求超时: {err}"
        print(e)
        return e
    except Exception as err:
        e = f"其他错误: {err}"
        print(e)
        return e

In [57]:
async def run_experiment(
    prompt: str, 
    temperature: float | None = None,
    top_p: float | None = None,
    do_sample: bool | None = None,
    count: int = 3,
    max_token: int = 1500,
    is_async: bool = False
) -> list[str]:
    """同一参数跑 n 次,返回 n 个输出文本。"""

    kwargs: dict = {
        "model": "glm-4.5-air",
        "messages": [
            {'role': 'user', 'content': prompt},
        ],
        "stream": False,
        "max_tokens": max_token # 不能超出给定的token数量
    }
    if temperature is not None:
        kwargs["temperature"] = temperature
    if top_p is not None:
        kwargs["top_p"] = top_p
    if do_sample is not None:
        kwargs["do_sample"] = do_sample

    if is_async: # 并发执行
        async with asyncio.TaskGroup() as tg: # asyncio.to_thread 将一个同步函数加入异步任务
            tasks = [tg.create_task(asyncio.to_thread(fetch, kwargs)) for _ in range(count)]
        return [task.result() for task in tasks]
    else: # 同步执行
        results = []
        for i in range(count):
            result = fetch(kwargs)
            results.append(result)
        return results

In [55]:
async def run_params(
    temperature_list: list[float] | None = None, 
    top_p_list: list[float] | None = None, 
    do_sample: bool | None = None,
    prompt: str = "中国的首都是"
):
    # temperature
    results: dict[float:list[str]] = {}

    if temperature_list is not None:
        for t in temperature_list:
            result = await run_experiment(prompt, temperature=t, do_sample=do_sample)
            results[f"{t}-do_sample_{do_sample}" if do_sample is not None else t] = result
    elif top_p_list is not None:
        for p in top_p_list:
            result = await run_experiment(prompt, top_p=p, do_sample=do_sample)
            results[f"{p}-do_sample_{do_sample}" if do_sample is not None else p] = result
    elif do_sample is not None:
        result = await run_experiment(prompt, do_sample=do_sample)
        results[f"do_sample_{do_sample}"] = result

    # 表格, 字典的键（key）作为列名，值（value）作为列的数据：
    df = pd.DataFrame(results)
    df.insert(loc=0, column="次数", value=[f"第{i+1}次" for i in range(len(df))])
    print(df)

In [58]:
await run_params(temperature_list=[0.0])

    次数  \
0  第1次   
1  第2次   
2  第3次   

                                                                                                                                                                                0.0  
0  中国的首都是**北京**（Beijing）。  \n\n北京是中国的政治、文化、国际交往和科技创新中心，拥有悠久的历史和丰富的文化遗产，如故宫、长城、天坛等世界著名景点。自1949年中华人民共和国成立后，北京一直作为首都，见证了中国的发展与变迁。  \n\n**小知识**：北京在历史上曾是元朝（大都）、明朝、清朝等多个朝代的首都，被誉为“千年古都”。  
1                   中国的首都是**北京**（Beijing）。  \n\n北京是中华人民共和国的政治、文化、国际交往和科技创新中心，拥有悠久的历史和丰富的文化遗产，如故宫、长城、天坛等世界著名景点。自1949年中华人民共和国成立后，北京一直作为首都，见证了中国的发展与变迁。  \n\n如果需要了解更多关于北京的信息，欢迎随时提问！ 😊  
2  中国的首都是**北京**（Běijīng）。  \n\n北京是中国的政治、文化、国际交往和科技创新中心，拥有悠久的历史和丰富的文化遗产，如故宫、天坛、长城等著名景点。作为首都，北京也是国家最高权力机关（如全国人民代表大会、国务院）的所在地。  \n\n🌟 **小知识**：北京自元朝起多次成为首都，如今是现代化国际大都市，同时完美融合了传统与现代风貌。  


In [53]:
await run_params(top_p_list=[0.1, 0.5, 1.0])

    次数                                    0.1  \
0  第1次  大海是流动的蔚蓝，以浪花的低语拥抱天地，用深邃的胸怀藏纳星辰与未知的远方。   
1  第2次  大海是流动的蔚蓝，以浪花的低语拥抱天地，用深邃的胸怀藏纳星辰与未知的远方。   
2  第3次        大海是蔚蓝的永恒，以潮汐为呼吸，托起日月星辰，也容纳万千生命。   

                                  0.5  \
0  大海是蔚蓝的胸怀，以潮汐的呼吸吞吐星辰，用无垠的深邃容纳浪花与远方。   
1           大海是蔚蓝的胸怀，以潮汐吞吐日月，用深邃包容万物。   
2       大海是无垠的蔚蓝，翻涌着不息的浪潮，深邃里藏着星辰与永恒。   

                                        1.0  
0  大海是天地间流动的蔚蓝诗篇，以浪花为笔，潮汐为韵，书写着永恒的深邃与生命的呼吸。  
1              大海是无垠的蓝绸，裹着浪花的呼吸，藏着星辰与深海的絮语。  
2                   大海是蔚蓝的深渊，盛着潮汐的呼吸与星辰的倒影。  


In [35]:
await run_params(do_sample=True)

    次数                  do_sample_True
0  第1次  大海是广袤的蔚蓝波涛，以永恒的呼吸吞吐着天地间的深邃与包容。
1  第2次         大海是流动的蔚蓝，裹挟着星辰与浪花的永恒呼吸。
2  第3次            大海是蔚蓝的摇篮，托起日月，也孕育生命。


In [40]:
await run_params(do_sample=False, temperature_list=[1.0])

    次数                              1.0-do_sample_False
0  第1次                  大海是无垠的蔚蓝怀抱，裹着浪花的呼吸，向天际铺展着生命的诗行。
1  第2次                       大海是天地揉碎的蓝，潮起时奔涌星河，落日里熔金万顷。
2  第3次  大海是蔚蓝的永恒诗篇，潮汐是它起伏的呼吸，浪花是它写给天空的信，藏着星辰与万千生命的深邃故事。
